In [1]:
from IPython.display import display, HTML
display(HTML('<style>.container { width:80% !important; }</style>'))

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import math
import urllib.parse

In [3]:
pd.set_option('display.max_columns', 500) # To utilise larger part of screen

pd.set_option('display.max_colwidth', None) # To show full cell text

In [4]:
data_path = 'data/cb_20240203'

# Data Import

In [8]:
organisations = pd.read_csv(data_path + '/organizations.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'status', 'short_description', 'category_list', 'category_groups_list', 'num_funding_rounds', 'total_funding_usd', 'founded_on', 'logo_url']]
organisations = organisations.rename(columns = {'uuid': 'org_uuid', 'name': 'org_name', 'country_code': 'org_country_code', 'region': 'org_region', 'city': 'org_city', 'logo_url': 'org_logo_url'})

organisations = organisations.assign(founded_on = pd.to_datetime(organisations['founded_on'], errors = 'coerce'))

organisations.head()

,org_uuid,org_name,org_country_code,org_region,org_city,status,short_description,category_list,category_groups_list,num_funding_rounds,total_funding_usd,founded_on,org_logo_url
0,e1393508-30ea-8a36-3f96-dd3226033abd,Wetpaint,USA,New York,New York,acquired,Wetpaint offers an online social publishing platform that helps digital publishers grow their customer base.,"Publishing,Social Media,Social Media Management","Content and Publishing,Internet Services,Media and Entertainment,Sales and Marketing",3.0,3.975000e+07,2005-06-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180177/2036b3394a37152e0ff69f27c71bc883.jpg
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Zoho,USA,California,Pleasanton,operating,"Zoho offers a suite of business, collaboration, and productivity applications.","Cloud Computing,Collaboration,Developer Tools,Enterprise Software,Information Services,Information Technology,Network Security,Project Management,Software,Web Apps","Administrative Services,Apps,Information Technology,Internet Services,Other,Privacy and Security,Software",NaN,NaN,1996-03-17,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180181/f8aaab73f17af0296eba5deda7a5b95b.png
2,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,Digg,USA,New York,New York,acquired,"Digg Inc. operates a website that enables its users to find, read, and share the most interesting and talked about stories on the internet.","Internet,Social Media,Social Network","Internet Services,Media and Entertainment",6.0,4.900000e+07,2004-10-11,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180182/e77f8f561153ffb45a9ffd538978380d.jpg
3,f4d5ab44-058b-298b-ea81-380e6e9a8eec,Omidyar Network,USA,California,Redwood City,operating,Omidyar Network is an investment firm.,"Enterprise Software,Financial Services,Venture Capital","Financial Services,Lending and Investments,Software",NaN,NaN,2004-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
4,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,USA,California,Menlo Park,ipo,"Meta is a social technology company that enables people to connect, find communities, and grow businesses.","Augmented Reality,Metaverse,Mortgage,Social Media,Social Network,Virtual Reality","Hardware,Internet Services,Media and Entertainment,Real Estate,Software",14.0,2.460782e+10,2004-02-04,https://images.crunchbase.com/image/upload/t_cb-default-original/whm4ed1rrc8skbdi3biv


In [7]:
people = pd.read_csv(data_path + '/people.csv')[['uuid', 'name', 'country_code', 'region', 'city', 'featured_job_organization_uuid', 'featured_job_title', 'logo_url']]
people = people.rename(columns = {'uuid': 'person_uuid', 'name': 'person_name', 'country_code': 'person_country_code', 'region': 'person_region', 'city': 'person_city', 'featured_job_organization_uuid': 'featured_job_org_uuid', 'logo_url': 'person_logo_url'})

people.head()

,person_uuid,person_name,person_country_code,person_region,person_city,featured_job_org_uuid,featured_job_title,person_logo_url
0,ed13cd36-fe2b-3707-197b-0c2d56e37a71,Ben Elowitz,USA,Washington,Seattle,1d845b32-7d80-47af-957d-78ccbeeaefb6,Co-Founder,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg
1,5ceca97b-493c-1446-6249-5aaa33464763,Kevin Flaherty,USA,Washington,Mercer Island,789e5e4d-0c90-d06e-92a0-b800b461c3da,Team Member,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180225/a307ba697e0f042623f05f35477bf495.jpg
2,9f99a98a-aa97-b30b-0d36-db67c1d277e0,Raju Vegesna,USA,California,San Francisco,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Chief Evangelist,https://images.crunchbase.com/image/upload/t_cb-default-original/oaxfoww3m0u0lwdovzv3
3,6e1bca72-a865-b518-b305-31214ce2d1b0,Ian Wenig,NaN,NaN,NaN,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,VP Business Development,https://images.crunchbase.com/image/upload/t_cb-default-original/v1442309935/yiajbpfbxyopc5l4zs4t.png
4,3b598c59-7b6c-2d48-763c-da55bca77035,Owen Byrne,USA,California,Mountain View,NaN,NaN,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180229/eb6f898475c0bef4c624ed44c9add19a.jpg


In [12]:
investors = pd.read_csv(data_path + '/investors.csv')[['uuid', 'name', 'roles', 'country_code', 'investor_types', 'investment_count', 'logo_url']]
investors = investors.rename(columns = {'uuid': 'person_uuid', 'name': 'investor_name', 'country_code': 'investor_country_code', 'logo_url': 'investor_logo_url'})

investors.head()

,person_uuid,investor_name,roles,investor_country_code,investor_types,investment_count,investor_logo_url
0,ed13cd36-fe2b-3707-197b-0c2d56e37a71,Ben Elowitz,investor,USA,angel,2.0,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180224/56303d2f4b99dd1dcc8abf17ba3fd8bf.jpg
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Zoho,"investor,company",USA,NaN,9.0,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180181/f8aaab73f17af0296eba5deda7a5b95b.png
2,f4d5ab44-058b-298b-ea81-380e6e9a8eec,Omidyar Network,"investor,company",USA,family_investment_office,343.0,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
3,df662812-7f97-0b43-9d3e-12f64f504fbb,Meta,"investor,company",USA,NaN,54.0,https://images.crunchbase.com/image/upload/t_cb-default-original/whm4ed1rrc8skbdi3biv
4,a01b8d46-d311-3333-7c34-aa3ae9c03f22,Mark Zuckerberg,investor,USA,"investment_partner,angel",8.0,https://images.crunchbase.com/image/upload/t_cb-default-original/v1448830269/gzcifut4c2xah95x0ewd.jpg


In [10]:
jobs = pd.read_csv(data_path + '/jobs.csv')[['uuid', 'person_uuid', 'org_uuid', 'started_on', 'ended_on', 'is_current', 'title', 'job_type']]
jobs = jobs.rename(columns = {'uuid': 'job_uuid', 'title': 'job_title'})

jobs = jobs.assign(started_on = pd.to_datetime(jobs['started_on'], errors = 'coerce'))
jobs = jobs.assign(ended_on = pd.to_datetime(jobs['ended_on'], errors = 'coerce'))

jobs.head()

,job_uuid,person_uuid,org_uuid,started_on,ended_on,is_current,job_title,job_type
0,697b6934-fc1f-9d63-cfb2-1a10759b378e,ed13cd36-fe2b-3707-197b-0c2d56e37a71,e1393508-30ea-8a36-3f96-dd3226033abd,2005-10-01,2014-06-01,False,Co-Founder and CEO,executive
1,b1de3765-442e-b556-9304-551c2a055901,5ceca97b-493c-1446-6249-5aaa33464763,e1393508-30ea-8a36-3f96-dd3226033abd,NaT,NaT,False,VP Marketing,executive
2,1319cd30-f5e8-c700-0af6-64029c6f7124,9f99a98a-aa97-b30b-0d36-db67c1d277e0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2000-11-01,NaT,True,Chief Evangelist,employee
3,27a252de-1ea8-c620-b2d4-5b889fa9b40f,6e1bca72-a865-b518-b305-31214ce2d1b0,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,2006-03-01,NaT,True,VP Business Development,executive
4,5a802a79-229f-44ae-0aba-db330f10b67a,c92a1f00-8c19-bf2e-0f28-dbbd383dc968,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,2005-07-01,2010-04-05,False,CEO,executive


In [11]:
funding_rounds = pd.read_csv(data_path + '/funding_rounds.csv')[['uuid', 'investment_type', 'announced_on', 'raised_amount_usd', 'org_uuid']]

funding_rounds = funding_rounds.rename(columns = {'uuid': 'funding_round_uuid'})

funding_rounds.head()

,funding_round_uuid,investment_type,announced_on,raised_amount_usd,org_uuid
0,8a945939-18e0-cc9d-27b9-bf33817b2818,angel,2004-09-01,500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
1,d950d7a5-79ff-fb93-ca87-13386b0e2feb,series_a,2005-05-01,12700000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
2,6fae3958-a001-27c0-fb7e-666266aedd78,series_b,2006-04-01,27500000.0,df662812-7f97-0b43-9d3e-12f64f504fbb
3,bcd5a63d-ed99-6963-0dd2-e36f6582f846,series_b,2006-05-01,10500000.0,f53cb4de-236e-0b1b-dee8-7104a8b018f9
4,60e6afd9-1215-465a-dd17-0ed600d4e29b,series_a,2007-01-17,NaN,4111dc8b-c0df-2d24-ed33-30cd137b3098


In [13]:
investments = pd.read_csv(data_path + '/investments.csv')[['uuid', 'funding_round_uuid', 'investor_uuid']]

investments = investments.rename(columns = {'uuid': 'investment_uuid'})

investments.head()

,investment_uuid,funding_round_uuid,investor_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,b08efc27-da40-505a-6f9d-c9e14247bf36
1,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,e2006571-6b7a-e477-002a-f7014f48a7e3
2,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,8d5c7e48-82da-3025-dd46-346a31bab86f
3,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,7ca12f7a-2f8e-48b4-a8d1-1a33a0e275b9
4,581c4b38-9653-7117-9bd4-7ffe5c7eba69,60e6afd9-1215-465a-dd17-0ed600d4e29b,fb2f8884-ec07-895a-48d7-d9a9d4d7175c


In [14]:
investment_partners = pd.read_csv(data_path + '/investment_partners.csv')[['uuid', 'funding_round_uuid', 'partner_uuid']]

investment_partners = investment_partners.rename(columns = {'uuid': 'investment_partner_uuid'})

investment_partners.head()

,investment_partner_uuid,funding_round_uuid,partner_uuid
0,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,2d78d1e7-203c-3eb6-bf1b-c51f10e0679b
1,524986f0-3049-54a4-fa72-f60897a5e61d,d950d7a5-79ff-fb93-ca87-13386b0e2feb,eaf6c243-d355-32f3-e23a-2a5fc82e8b34
2,6556ab92-6465-25aa-1ffc-7f8b4b09a476,6fae3958-a001-27c0-fb7e-666266aedd78,478e7efd-bec4-b9f5-304b-cffedc1fc012
3,0216e06a-61f8-9cf1-19ba-20811229c53e,6fae3958-a001-27c0-fb7e-666266aedd78,0f9f3c05-cb79-f58f-6cc5-98ddc8382d4f
4,dadd7d86-520d-5e35-3033-fc1d8792ab91,bcd5a63d-ed99-6963-0dd2-e36f6582f846,ea9f4980-600c-84f4-a5d6-b4f8c2f787fb


# Defining Constants

In [15]:
# Lowest amount of company funding to consider
funding_cutoff = 100000000

# Company to find alumns for
target_org_uuid = '96ab87ca-00b5-2ebc-f218-86262954e320' # PayPal

# Years after foundation to consider
years_founding_cutoff = 3

# Creating Main Dataset

Filter organisations on target name

## target_org - Information about the target organisation
Fields:
- org_uuid
- org_name
- org_country_code
- org_region
- org_city
- short_description
- category_list
- total_funding_usd
- founded_on


## alumni_info - Information about target organisation alumnis
Fields:
- person_uuid
- person_name
- job_title
- job_type (founder, executive, investor)
- person_logo_url

Process:
1. Filter jobs on:
    - jobs['org_uuid'] == target_org['org_uuid]
    - jobs['job_type'] == 'executive'
    - jobs['started_on'] < target_org['founded_on'] + years_founding_cutoff
    - jobs['started_on'] >= target_org['founded_on']
<br/><br/>
2. Left merge with people on person_uuid

## subsequent_orgs - Organisations the alumni of the target organisation are associated with
Fields:
- person_uuid
- subsequent_org_uuid
- subsequent_org_name
- subsequent_position_type (founder, executive, investor, board member)
- funding_amount_usd

Process:
1. Left merge alumni_info with jobs on person_uuid for subsequent_orgs_jobs
2. Filter on:
    - subsequent_orgs_jobs['job_type'].isin(['executive', 'board_member', 'advisor'])
<br/><br/>
3. Left merge investments with funding_rounds on funding_round_uuid for investments_info
4. Inner merge alumni_info with investments_info on person_uuid and investor_uuid for subsequent_orgs_investments # TODO also add partner investments at VCs
5. Concat subsequent_orgs_jobs with subsequent_orgs_investments for subsequent_orgs
6. Left merge subsequent_orgs_investments with organisations on org_uuid for final subsequent_orgs_investments

## target_org - Information about the target organisation

In [17]:
target_org = organisations.loc[organisations['org_uuid'] == target_org_uuid][['org_uuid', 'org_name', 'org_country_code', 'org_region', 'org_city', 'short_description', 'category_list', 'total_funding_usd', 'founded_on', 'org_logo_url']]

target_org.head()

,org_uuid,org_name,org_country_code,org_region,org_city,short_description,category_list,total_funding_usd,founded_on,org_logo_url
359,96ab87ca-00b5-2ebc-f218-86262954e320,PayPal,USA,California,San Jose,PayPal is a financial service company that provides online payment solutions to its users worldwide.,"E-Commerce Platforms,FinTech,Mobile Payments,Transaction Processing",5.216000e+09,1998-12-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1447476896/lcrdrlh1hfa0nbl9ycy3.png


In [18]:
export_df = target_org.rename(columns={
    'org_uuid': 'orgUuid',
    'org_name': 'orgName',
    'org_country_code': 'orgCountryCode',
    'org_region': 'orgRegion',
    'org_city': 'orgCity',
    'short_description': 'shortDescription',
    'category_list': 'categoryList',
    'total_funding_usd': 'totalFundingUsd',
    'founded_on': 'foundedOn',
    'org_logo_url': 'orgLogoUrl'
}).drop(columns = ['categoryList'], axis = 1)

export_df.to_json('data/output/targetOrg.json', orient = 'records')

## alumni_info - Information about target organisation alumnis

In [19]:
executives_target = jobs.loc[
        (jobs['org_uuid'] == target_org_uuid)
        & (jobs['job_type'] == 'executive')
        & (jobs['started_on'] < target_org['founded_on'].values[0] + pd.DateOffset(years = years_founding_cutoff))
        & (jobs['started_on'] >= target_org['founded_on'].values[0])
]

alumni_info = pd.merge(
    executives_target,
    people,
    how = 'left',
    on = 'person_uuid'
)[['person_uuid', 'person_name', 'job_title', 'job_type', 'started_on', 'ended_on', 'person_logo_url']]

alumni_info.head()

,person_uuid,person_name,job_title,job_type,started_on,ended_on,person_logo_url
0,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,executive,2000-01-01,2009-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg
1,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",executive,2001-09-01,2004-10-01,https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1
2,35a76dc5-1298-dfbc-c432-acd9fdd70cbd,Russel Simmons,Lead Software Architect,executive,1999-01-01,2003-03-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180387/10da3e9f6bd911b14b7e8ab3243615a1.jpg
3,6bf64de8-275a-1161-ef03-5c7dc4d26dfd,Vince Sollitto,"Vice President, Corporate Communications and Public Affairs",executive,2000-01-01,2002-11-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397185801/9ff0830ae5da6d8655d1f0f50762902f.jpg
4,b0e4e511-a1f1-162d-005a-006a94ee35f6,David O. Sacks,COO,executive,1999-01-01,2002-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1477024172/omx6g1wq1ygyybzvxzga.png


In [22]:
export_df = alumni_info.drop_duplicates(subset = ['person_uuid'])[['person_uuid', 'person_name', 'job_title', 'job_type', 'started_on', 'ended_on', 'person_logo_url']].rename(columns={
    'person_uuid': 'personUuid',
    'person_name': 'personName',
    'job_title': 'jobTitle',
    'job_type': 'jobType',
    'started_on': 'startedOn',
    'ended_on': 'endedOn',
    'person_logo_url': 'personLogoUrl',
})

export_df.to_json('data/output/alumniInfo.json', orient = 'records')

In [22]:
export_df = alumni_info.drop_duplicates(subset = ['person_uuid'])[['person_uuid', 'person_name', 'person_logo_url']].rename(columns={
    'person_uuid': 'personUuid',
    'person_name': 'personName',
    'person_logo_url': 'personLogoUrl',
})

export_df.to_json('data/output/alumniLogos.json', orient = 'records')

## subsequent_orgs - Organisations the alumni of the target organisation are associated with

In [23]:
alumni_jobs = jobs.loc[
    jobs['person_uuid'].isin(alumni_info['person_uuid'])
    & jobs['job_type'].isin(['executive', 'board_member', 'advisor'])
]

subsequent_orgs_jobs = (
    pd.merge(alumni_info, alumni_jobs, how = 'left', on = 'person_uuid', suffixes = ['_target', '_subsequent'])
    .drop_duplicates(subset = ['person_uuid', 'job_uuid'])
).rename(columns = {'job_type_subsequent': 'relation_type'})

subsequent_orgs_jobs = subsequent_orgs_jobs.loc[subsequent_orgs_jobs['started_on_target'] < subsequent_orgs_jobs['started_on_subsequent']][['person_uuid', 'org_uuid', 'relation_type', 'job_title_subsequent']]

subsequent_orgs_jobs

,person_uuid,org_uuid,relation_type,job_title_subsequent
1,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,f4d5ab44-058b-298b-ea81-380e6e9a8eec,executive,Partner of Human Capital & Operations Functions
6,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Member of the Board of Directors
7,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Chair Of The Board Of Directors
8,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,692833eb-f832-428c-1c78-d693f0120373,advisor,"Member, GSAS (Graduate School of Arts and Sciences) Advisory Board"
9,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,b0ec4bed-8fcc-e20f-ccda-e9175b10a25d,board_member,Board Member
...,...,...,...,...
206,b1bdd955-0d70-68d4-6541-09a04f73ae4a,eef9eab2-4c50-f0a3-12b8-ce721fa2cc81,executive,SRE Manager
207,b1bdd955-0d70-68d4-6541-09a04f73ae4a,c18f68ff-0871-4ac8-8d66-6dfde639fce6,executive,"Director, Engineering"
209,f8b56ea9-9a0c-49ac-943e-b2b92080e004,0c867fde-2b9a-df10-fdb9-66b74f355f91,executive,"COO, Sequoia Capital Global Equities"
211,f8b56ea9-9a0c-49ac-943e-b2b92080e004,7bcc0a57-a5f6-7ef5-0945-28bcbfc54559,executive,COO


In [24]:
investments_info = pd.merge(
    investments,
    funding_rounds,
    how = 'left',
    on = 'funding_round_uuid'
)[['org_uuid', 'investor_uuid', 'investment_type', 'announced_on']]

subsequent_orgs_investments = (
    investments_info
    .loc[investments_info['investor_uuid'].isin(alumni_info['person_uuid'])]
    .assign(relation_type = 'investor')
    .assign(job_title_subsequent = 'investor')
    .rename(columns = {'investor_uuid': 'person_uuid'})
)[['person_uuid', 'org_uuid', 'relation_type', 'job_title_subsequent']]

subsequent_orgs_investments

,person_uuid,org_uuid,relation_type,job_title_subsequent
841,fe73e532-a1e3-7bbf-cc69-5df480427e51,9d92578e-a7df-6e08-296f-4f0be4191339,investor,investor
1561,3f47be49-2e32-8118-01a0-31685a4d0fd7,ff71401a-f5b0-1dd8-2b72-4a9e5d6ecba0,investor,investor
1611,3f47be49-2e32-8118-01a0-31685a4d0fd7,093b4f08-7be6-03f3-31f2-6a640386132d,investor,investor
2350,fe73e532-a1e3-7bbf-cc69-5df480427e51,d8df4d27-60ad-a762-cc42-d5ea83e754e4,investor,investor
2656,fe73e532-a1e3-7bbf-cc69-5df480427e51,89ea3dab-6e0d-84de-c2a4-28a26986e7d7,investor,investor
...,...,...,...,...
961600,3f47be49-2e32-8118-01a0-31685a4d0fd7,8528f6b2-fa66-5c35-ffee-8221371fed36,investor,investor
969636,3f47be49-2e32-8118-01a0-31685a4d0fd7,d94f8172-5a0b-624a-ee11-e752fb561c59,investor,investor
985265,3f47be49-2e32-8118-01a0-31685a4d0fd7,852d3b83-4f9b-4dd5-be95-c767291e3696,investor,investor
1012078,d3326bcc-6d25-9214-60c7-1e95c5f2f2a1,e0f90b0e-4ca2-415a-be44-f2c82bc78445,investor,investor


In [25]:
subsequent_orgs = pd.concat([subsequent_orgs_jobs, subsequent_orgs_investments])

subsequent_orgs.head()

,person_uuid,org_uuid,relation_type,job_title_subsequent
1,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,f4d5ab44-058b-298b-ea81-380e6e9a8eec,executive,Partner of Human Capital & Operations Functions
6,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Member of the Board of Directors
7,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Chair Of The Board Of Directors
8,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,692833eb-f832-428c-1c78-d693f0120373,advisor,"Member, GSAS (Graduate School of Arts and Sciences) Advisory Board"
9,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,b0ec4bed-8fcc-e20f-ccda-e9175b10a25d,board_member,Board Member


In [27]:
subsequent_orgs_info = (
    pd.merge(subsequent_orgs, organisations, how = 'left', on = 'org_uuid')
    .rename(columns = {'job_title': 'job_title_subsequent'})
)[['person_uuid', 'org_uuid', 'relation_type', 'job_title_subsequent', 'org_name', 'org_country_code', 'org_city', 'short_description', 'category_list', 'total_funding_usd', 'founded_on', 'org_logo_url']]

subsequent_orgs_info = subsequent_orgs_info.loc[subsequent_orgs_info['org_uuid'] != target_org['org_uuid'].values[0]]

subsequent_orgs_info

,person_uuid,org_uuid,relation_type,job_title_subsequent,org_name,org_country_code,org_city,short_description,category_list,total_funding_usd,founded_on,org_logo_url
0,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,f4d5ab44-058b-298b-ea81-380e6e9a8eec,executive,Partner of Human Capital & Operations Functions,Omidyar Network,USA,Redwood City,Omidyar Network is an investment firm.,"Enterprise Software,Financial Services,Venture Capital",NaN,2004-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
1,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Member of the Board of Directors,Global Innovation Fund,GBR,London,GIF invests in social innovations that aim to improve the lives and opportunities of millions of people in the developing world.,"Communities,Finance,Financial Services,Non Profit",NaN,2014-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/yxpwtz6tmnvjqotzdste
2,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Chair Of The Board Of Directors,Global Innovation Fund,GBR,London,GIF invests in social innovations that aim to improve the lives and opportunities of millions of people in the developing world.,"Communities,Finance,Financial Services,Non Profit",NaN,2014-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/yxpwtz6tmnvjqotzdste
3,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,692833eb-f832-428c-1c78-d693f0120373,advisor,"Member, GSAS (Graduate School of Arts and Sciences) Advisory Board",Fordham University,USA,New York,Fordham University offers education distinguished by the Jesuit tradition.,"Education,Higher Education,Performing Arts",NaN,1841-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/v1423282901/cwnrhg91boedreb1ob5m.jpg
4,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,b0ec4bed-8fcc-e20f-ccda-e9175b10a25d,board_member,Board Member,iMerit,USA,Los Gatos,"iMerit enriches and annotates the data that powers algorithms in Machine Learning, Computer Vision, and Natural Language Processing.","Analytics,Artificial Intelligence (AI),Computer Vision,Crowdsourcing,Customer Service,Image Recognition,Machine Learning,Natural Language Processing",36300000.0,2012-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/flttzjv4gqtee9yc7nv1
...,...,...,...,...,...,...,...,...,...,...,...,...
422,3f47be49-2e32-8118-01a0-31685a4d0fd7,8528f6b2-fa66-5c35-ffee-8221371fed36,investor,investor,Treasure Financial,USA,San Francisco,"Secure, High-Return, Liquid Cash Management.","Financial Services,FinTech",14000000.0,2021-01-27,https://images.crunchbase.com/image/upload/t_cb-default-original/insmdpglv0kn8jzcbcdi
423,3f47be49-2e32-8118-01a0-31685a4d0fd7,d94f8172-5a0b-624a-ee11-e752fb561c59,investor,investor,Luminar,USA,Orlando,"Luminar is an autonomous vehicle and lidar technology company for passenger cars, commercial trucking and robo-taxi.","Automotive,Autonomous Vehicles,Sensor,Software",779750000.0,2012-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/fpbzsyzjmlegyvqga7ix
424,3f47be49-2e32-8118-01a0-31685a4d0fd7,852d3b83-4f9b-4dd5-be95-c767291e3696,investor,investor,Reserve,USA,Oakland,Reserve is a financial services firm that offers digital currency.,Retail,5010000.0,2017-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/cgdhvz5m4mfchap5gqn1
425,d3326bcc-6d25-9214-60c7-1e95c5f2f2a1,e0f90b0e-4ca2-415a-be44-f2c82bc78445,investor,investor,BIOSORRA,USA,Asheville,Leading carbon removal climate-tech company with proprietary tech and R&D to address climate justice,Carbon Capture,335000.0,2022-01-01,https://images.crunchbase.com/image/upload/t_cb-default-original/jgirpcivopf9feznymo4


## relations - Relationships between target org people and subsequent org

In [28]:
relations = subsequent_orgs_info[['person_uuid', 'org_uuid', 'relation_type', 'job_title_subsequent', 'org_logo_url']]

relations

,person_uuid,org_uuid,relation_type,job_title_subsequent,org_logo_url
0,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,f4d5ab44-058b-298b-ea81-380e6e9a8eec,executive,Partner of Human Capital & Operations Functions,https://images.crunchbase.com/image/upload/t_cb-default-original/v1491524876/tjg7440r2uc0s5neswkp.png
1,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Member of the Board of Directors,https://images.crunchbase.com/image/upload/t_cb-default-original/yxpwtz6tmnvjqotzdste
2,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Chair Of The Board Of Directors,https://images.crunchbase.com/image/upload/t_cb-default-original/yxpwtz6tmnvjqotzdste
3,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,692833eb-f832-428c-1c78-d693f0120373,advisor,"Member, GSAS (Graduate School of Arts and Sciences) Advisory Board",https://images.crunchbase.com/image/upload/t_cb-default-original/v1423282901/cwnrhg91boedreb1ob5m.jpg
4,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,b0ec4bed-8fcc-e20f-ccda-e9175b10a25d,board_member,Board Member,https://images.crunchbase.com/image/upload/t_cb-default-original/flttzjv4gqtee9yc7nv1
...,...,...,...,...,...
422,3f47be49-2e32-8118-01a0-31685a4d0fd7,8528f6b2-fa66-5c35-ffee-8221371fed36,investor,investor,https://images.crunchbase.com/image/upload/t_cb-default-original/insmdpglv0kn8jzcbcdi
423,3f47be49-2e32-8118-01a0-31685a4d0fd7,d94f8172-5a0b-624a-ee11-e752fb561c59,investor,investor,https://images.crunchbase.com/image/upload/t_cb-default-original/fpbzsyzjmlegyvqga7ix
424,3f47be49-2e32-8118-01a0-31685a4d0fd7,852d3b83-4f9b-4dd5-be95-c767291e3696,investor,investor,https://images.crunchbase.com/image/upload/t_cb-default-original/cgdhvz5m4mfchap5gqn1
425,d3326bcc-6d25-9214-60c7-1e95c5f2f2a1,e0f90b0e-4ca2-415a-be44-f2c82bc78445,investor,investor,https://images.crunchbase.com/image/upload/t_cb-default-original/jgirpcivopf9feznymo4


In [29]:
export_df = relations.rename(columns={
    'person_uuid': 'personUuid',
    'org_uuid': 'orgUuid',
    'relation_type': 'relationType',
    'job_title_subsequent': 'jobTitleSubsequent',
    'org_logo_url': 'orgLogoUrl',
})

export_df.to_json('data/output/relations.json', orient = 'records')

## subsequent_orgs_info - Finalisation (only having org info in there)

In [30]:
subsequent_orgs_info = (
    subsequent_orgs_info
    .drop_duplicates(subset = ['org_uuid'])
)[['org_uuid', 'org_name', 'org_country_code', 'org_city', 'short_description', 'category_list', 'total_funding_usd', 'founded_on', 'org_logo_url']]

In [31]:
export_df = subsequent_orgs_info.rename(columns={
    'org_uuid': 'orgUuid',
    'org_name': 'orgName',
    'org_country_code': 'orgCountryCode',
    'org_city': 'orgCity',
    'short_description': 'shortDescription',
    'category_list': 'categoryList',
    'total_funding_usd': 'totalFundingUsd',
    'founded_on': 'foundedOn',
    'org_logo_url': 'orgLogoUrl',
}).drop(columns = ['categoryList'], axis = 1)

export_df.to_json('data/output/subsequentOrgsInfo.json', orient = 'records')

## alumni_network - Final DF with all info

In [19]:
alumni_network = (
    pd.merge(alumni_info, subsequent_orgs_info.drop_duplicates(), how = 'left', on = 'person_uuid')
)[['person_uuid', 'person_name', 'job_title', 'person_logo_url', 'org_uuid', 'relation_type', 'job_title_subsequent', 'org_name', 'org_country_code', 'category_list', 'total_funding_usd', 'founded_on']]

alumni_network

,person_uuid,person_name,job_title,person_logo_url,org_uuid,relation_type,job_title_subsequent,org_name,org_country_code,category_list,total_funding_usd,founded_on
0,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg,f4d5ab44-058b-298b-ea81-380e6e9a8eec,executive,Partner of Human Capital & Operations Functions,Omidyar Network,USA,"Enterprise Software,Financial Services,Venture Capital",NaN,2004-01-01
1,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Member of the Board of Directors,Global Innovation Fund,GBR,"Communities,Finance,Financial Services,Non Profit",NaN,2014-01-01
2,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg,0fa6834d-ae43-e879-4141-22a718b6ed0c,board_member,Chair Of The Board Of Directors,Global Innovation Fund,GBR,"Communities,Finance,Financial Services,Non Profit",NaN,2014-01-01
3,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg,692833eb-f832-428c-1c78-d693f0120373,advisor,"Member, GSAS (Graduate School of Arts and Sciences) Advisory Board",Fordham University,USA,"Education,Higher Education,Performing Arts",NaN,1841-01-01
4,8ab40f91-fd7f-1dac-3f2b-8c4c68c5a649,Salvatore Giambanco,Vice President of Human Resources & Administration,https://images.crunchbase.com/image/upload/t_cb-default-original/v1397180760/60e833f4bcd98df7a34b012b8a0c86a8.jpg,b0ec4bed-8fcc-e20f-ccda-e9175b10a25d,board_member,Board Member,iMerit,USA,"Analytics,Artificial Intelligence (AI),Computer Vision,Crowdsourcing,Customer Service,Image Recognition,Machine Learning,Natural Language Processing",36300000.0,2012-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...
488,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,10553713-d4c1-4e52-bfac-688eaed58d91,investor,investor,Arkam Intelligence,USA,"Artificial Intelligence (AI),Bitcoin,Information Technology",12000000.0,2020-01-01
489,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,5385633c-d5ed-460b-87eb-9b32ef2f8fa8,investor,investor,Whop,USA,"E-Commerce,Internet,Marketplace",18000002.0,2021-03-01
490,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,08aa496a-2218-46f2-ae70-7182c2c0a2b3,investor,investor,Alias Technologies,USA,"Consulting,Financial Services",3000000.0,NaT
491,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,852d3b83-4f9b-4dd5-be95-c767291e3696,investor,investor,Reserve,USA,Retail,5010000.0,2017-01-01


In [20]:
return_df = alumni_network.loc[alumni_network['total_funding_usd'] >= funding_cutoff]

return_df

,person_uuid,person_name,job_title,person_logo_url,org_uuid,relation_type,job_title_subsequent,org_name,org_country_code,category_list,total_funding_usd,founded_on
6,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1,b2d5980b-bccb-a961-f0ac-6e4e46b30ede,board_member,Member of the Board of Directors,Twilio,USA,"Enterprise Software,Messaging,Mobile Apps,SMS,Software",614415525.0,2008-06-30
7,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1,38949e1e-7e1f-2611-a2af-865dfdc94130,advisor,Member of the Advisory Board,Life360,USA,"Android,Apps,Family,Mobile,Mobile Apps",140236268.0,2008-01-01
43,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1,5112119f-6bb7-4417-8d55-a42295ec9277,investor,investor,Goldbelly,USA,"Delivery Service,E-Commerce,Food and Beverage,Last Mile Transportation",133120000.0,2012-01-01
50,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1,ab155857-2dc2-e510-9f3c-81be7897feb4,investor,investor,Breather,CAN,Real Estate,134600000.0,2012-11-01
67,fe73e532-a1e3-7bbf-cc69-5df480427e51,Dave McClure,"Director, Marketing",https://images.crunchbase.com/image/upload/t_cb-default-original/fi7bfp6l7yubkba7xdj1,03ef8031-2efb-4fa6-0648-1b09ca50c1ab,investor,investor,Tribal,USA,"Credit Cards,Finance,Financial Services,FinTech",292500000.0,2016-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...
463,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,7fe366df-3f8c-4ef6-a9a5-3f093e6142d0,investor,investor,BitDAO,PAN,"Bitcoin,Cryptocurrency,Finance",230000000.0,2021-01-01
464,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,bd3ec59b-ba2b-4ec2-922d-25b2e77794e2,investor,investor,TMRW Life Sciences,USA,"Biotechnology,Health Care,Life Science,Software",153500000.0,2018-01-01
468,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,7b17f8af-4b1a-4663-880d-71ee2d352413,investor,investor,Hallow,USA,"Apps,Health Care,Religion,Wellness",105000000.0,2018-06-21
474,3f47be49-2e32-8118-01a0-31685a4d0fd7,Peter Thiel,"CEO, Chairman, and Co-Founder",https://images.crunchbase.com/image/upload/t_cb-default-original/t2pgpngl4duo3tnqtjyc,4ea45d69-6587-4fe9-b20d-f80cfe4188e2,investor,investor,Cleerly,USA,"Apps,Artificial Intelligence (AI),Health Care,Medical,Wellness",280484975.0,2017-01-01


In [21]:
export_df = return_df.rename(columns={
    'person_uuid': 'personUuid',
    'person_name': 'personName',
    'job_title': 'jobTitle',
    'person_logo_url': 'personLogoUrl',
    'org_uuid': 'orgUuid',
    'relation_type': 'relationType',
    'job_title_subsequent': 'jobTitleSubsequent',
    'org_name': 'orgName',
    'org_country_code': 'orgCountryCode',
    'category_list': 'categoryList',
    'total_funding_usd': 'totalFundingUsd',
    'founded_on': 'foundedOn',
}).drop(columns = ['categoryList'], axis = 1)

export_df.to_json('data/output/alumni_network.json', orient = 'records')